In [ ]:
!pip install google-play-scraper transformers torch scikit-learn wordcloud

import pandas as pd
import numpy as np
import torch
import time
import re
import matplotlib.pyplot as plt

from google_play_scraper import reviews, Sort
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from wordcloud import WordCloud
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
## SCRAPING DATA
APP_ID = 'com.bibit.bibitid'
BATCH_SIZE = 200
MAX_DATA = 3000

all_reviews = []
continuation_token = None

while len(all_reviews) < MAX_DATA:
    result, continuation_token = reviews(
        APP_ID,
        lang='id',
        country='id',
        sort=Sort.NEWEST,
        count=BATCH_SIZE,
        continuation_token=continuation_token
    )

    if not result:
        break

    all_reviews.extend(result)
    print(f"Total: {len(all_reviews)}")
    time.sleep(1)

df = pd.DataFrame(all_reviews)
df = df[['content', 'score']]
df.columns = ['review', 'rating']

print("Jumlah data awal:", len(df))

In [ ]:
## CLEANING MINIMAL
df = df[['review', 'rating']].dropna()

# buang review terlalu pendek
df = df[df['review'].str.len() > 5]

df = df.reset_index(drop=True)

print("Jumlah setelah cleaning:", len(df))

In [ ]:
df

In [ ]:
## LABELING SENTIMEN
def label_sentiment(rating):
    if rating <= 2:
        return 0  # negatif
    elif rating == 3:
        return 1  # netral
    else:
        return 2  # positif

df['label'] = df['rating'].apply(label_sentiment)

In [ ]:
## SPLIT DATA
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['review'],
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

In [ ]:
## LOAD INDOBERT
model_name = "indobenchmark/indobert-base-p1"

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
## TOKENIZATION
train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True,
    max_length=128
)

test_encodings = tokenizer(
    list(test_texts),
    truncation=True,
    padding=True,
    max_length=128
)

In [ ]:
## DATASET CLASS
class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = Dataset(train_encodings, train_labels)
test_dataset = Dataset(test_encodings, test_labels)

In [ ]:
## LOAD MODEL
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

In [ ]:
## TRAINING SETUP
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc}

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    logging_dir='./logs',
    save_strategy="epoch",
    load_best_model_at_end=True
)

In [ ]:
## TRAIN MODEL
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
## EVALUASI MODEL
predictions = trainer.predict(test_dataset)

y_pred = np.argmax(predictions.predictions, axis=1)
y_true = test_labels

print("Accuracy:", accuracy_score(y_true, y_pred))
print("\nClassification Report:\n", classification_report(y_true, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_true, y_pred))

In [ ]:
## PREDIKSI CONTOH
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = model(**inputs)
    probs = torch.nn.functional.softmax(outputs.logits, dim=1)
    return torch.argmax(probs).item()

print(predict_sentiment("aplikasi ini lama banget dan sering error"))

# EDA

In [ ]:
## CLEANING UNTUK ANALISIS (STOPWORD)

import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

stop_words = set(stopwords.words('indonesian'))

# tambahan
custom_stopwords = ['nya', 'aja', 'banget', 'sih', 'dong']
stop_words.update(custom_stopwords)

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

df['clean_review'] = df['review'].apply(clean_text)

In [ ]:
## DISTRIBUSI RATING
df['rating'].value_counts().sort_index().plot(kind='bar')
plt.title("Distribusi Rating")
plt.show()

In [ ]:
## DATA NEGATIF
df_neg = df[df['rating'] <= 2]
print("Jumlah data negatif:", len(df_neg))

In [ ]:
## WORDCLOUD
from wordcloud import WordCloud

text_neg = " ".join(df_neg['clean_review'])

wc = WordCloud(width=800, height=400, background_color='white').generate(text_neg)

plt.imshow(wc)
plt.axis("off")
plt.title("Wordcloud Keluhan (Negatif)")
plt.show()

In [ ]:
## BIGRAM
from sklearn.feature_extraction.text import CountVectorizer

def get_top_bigrams(corpus, n=15):
    vec = CountVectorizer(ngram_range=(2,2)).fit(corpus)
    bag = vec.transform(corpus)
    sum_words = bag.sum(axis=0)

    words_freq = [(word, sum_words[0, idx])
                  for word, idx in vec.vocabulary_.items()]

    return sorted(words_freq, key=lambda x: x[1], reverse=True)[:n]

top_bigrams = get_top_bigrams(df_neg['clean_review'])

print("\nTop Bigrams:")
for i in top_bigrams:
    print(i)

In [ ]:
## TRIGRAM
def get_top_trigrams(corpus, n=10):
    vec = CountVectorizer(ngram_range=(3,3)).fit(corpus)
    bag = vec.transform(corpus)
    sum_words = bag.sum(axis=0)

    words_freq = [(word, sum_words[0, idx])
                  for word, idx in vec.vocabulary_.items()]

    return sorted(words_freq, key=lambda x: x[1], reverse=True)[:n]

top_trigrams = get_top_trigrams(df_neg['clean_review'])

print("\nTop Trigrams:")
for i in top_trigrams:
    print(i)

# Visualisasi

In [ ]:
## CONFUSION MATRIX
import seaborn as sns

cm = confusion_matrix(y_true, y_pred)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negatif','Netral','Positif'],
            yticklabels=['Negatif','Netral','Positif'])

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
## BIGRAM DOMINAN PADA ULASAN NEGATIF
words = [w[0] for w in top_bigrams]
counts = [w[1] for w in top_bigrams]

plt.barh(words, counts)
plt.title("Top Bigrams Keluhan")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
## DISTRIBUSI SENTIMEN
pd.Series(y_pred).value_counts().plot(kind='bar')
plt.title("Distribusi Hasil Prediksi Sentimen")
plt.show()